In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer('all-MiniLM-L6-v2')


/Users/evan/Projects/llm-zoomcamp-2026-code/module_02_vector_search/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7049.94it/s]


In [3]:
vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [4]:
#We always have to encode thw question the user is asking
query = "Yo! I just discovered the course? Is it still open?"

query_vector = model.encode(inputs=query)

results = vs_index.search(
    query_vector=query_vector,
    filter_dict={'course':'llm-zoomcamp'}, 
    num_results=5
    )

In [5]:
results

[{'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': '977bf7786c',
  

## RAG Calls on Persistent Storage

In [6]:
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [7]:
from openai import OpenAI

openai_client = OpenAI()

In [8]:
vector_assistant = RAGVector(
    embedder=model, 
    index=vs_index,
    llm_client=openai_client
)

In [9]:
vector_assistant.rag(query)

'Yes, you can still join. But if you want a certificate, you need to submit your project while submissions are still open.'

In [22]:
# Start here: https://youtu.be/0P54MFyz-mc?si=A3Az6XJELEoMTnxf 